In [1]:
import cv2
import os
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.preprocessing import StandardScaler, LabelEncoder, normalize
from sklearn.svm import SVC
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy

# -------------------------
# MODEL
# -------------------------
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
model.classifier = torch.nn.Identity()
model.eval()

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# -------------------------
# FEATURE FUNCTIONS
# -------------------------
def extract_deep_features(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = transform(img).unsqueeze(0)
    with torch.no_grad():
        features = model(tensor)
    return features.numpy().flatten()

def extract_handcrafted_features(img):

    img = cv2.resize(img,(256,256))

    hsv = cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    s = hsv[:,:,1]
    _,mask = cv2.threshold(s,30,255,cv2.THRESH_BINARY)

    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel)

    contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    if len(contours)==0:
        return None

    cnt=max(contours,key=cv2.contourArea)

    area=cv2.contourArea(cnt)
    perimeter=cv2.arcLength(cnt,True)

    circularity=(4*np.pi*area)/(perimeter**2+1e-6)

    hull=cv2.convexHull(cnt)
    solidity=area/(cv2.contourArea(hull)+1e-6)

    x,y,w,h=cv2.boundingRect(cnt)
    aspect_ratio=w/h
    extent=area/(w*h+1e-6)

    R,G,B=cv2.split(img)

    R_mean,R_std=cv2.meanStdDev(R,mask=mask)
    G_mean,G_std=cv2.meanStdDev(G,mask=mask)
    B_mean,B_std=cv2.meanStdDev(B,mask=mask)

    H,S,V=cv2.split(hsv)

    H_mean,H_std=cv2.meanStdDev(H,mask=mask)
    S_mean,S_std=cv2.meanStdDev(S,mask=mask)
    V_mean,V_std=cv2.meanStdDev(V,mask=mask)

    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)

    L,A,B_lab=cv2.split(lab)

    L_mean,L_std=cv2.meanStdDev(L,mask=mask)
    a_mean,a_std=cv2.meanStdDev(A,mask=mask)
    b_mean,b_std=cv2.meanStdDev(B_lab,mask=mask)

    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    gray_masked=cv2.bitwise_and(gray,gray,mask=mask)

    laplacian=cv2.Laplacian(gray_masked,cv2.CV_64F).var()

    glcm=graycomatrix(gray_masked,[1],[0],256,symmetric=True,normed=True)

    contrast=graycoprops(glcm,'contrast')[0,0]
    energy=graycoprops(glcm,'energy')[0,0]
    homogeneity=graycoprops(glcm,'homogeneity')[0,0]

    hist=cv2.calcHist([gray_masked],[0],mask,[256],[0,256])
    hist_norm=hist/hist.sum()

    ent=entropy(hist_norm.flatten())

    dark_pixel_ratio=np.sum(gray_masked<50)/(np.sum(mask>0)+1e-6)

    return [R_mean[0][0],G_mean[0][0],B_mean[0][0],
            R_std[0][0],G_std[0][0],B_std[0][0],
            H_mean[0][0],S_mean[0][0],V_mean[0][0],
            H_std[0][0],S_std[0][0],V_std[0][0],
            L_mean[0][0],L_std[0][0],
            a_mean[0][0],a_std[0][0],
            b_mean[0][0],b_std[0][0],
            laplacian,contrast,energy,homogeneity,
            ent,
            area,perimeter,circularity,solidity,
            aspect_ratio,extent,dark_pixel_ratio]

# -------------------------
# DATASET PATH
# -------------------------
dataset_path = r"C:\Users\KIIT0001\Downloads\archive (6)\vegetable_Dataset"

data = []
labels = []

print("Loading dataset...")

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    for img_name in os.listdir(folder_path):

        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        hand_feat = extract_handcrafted_features(img)
        if hand_feat is None:
            continue

        deep_feat = extract_deep_features(img)

        fused = np.concatenate((hand_feat, deep_feat))

        data.append(fused)
        labels.append(folder)

print("Dataset Loaded")

# -------------------------
# TRAIN MODEL
# -------------------------
X_raw = np.array(data)
y_raw = np.array(labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_norm = normalize(X_scaled)

svm_model = SVC(kernel='rbf', probability=True)
svm_model.fit(X_norm, y_encoded)

print("Training Done")

# -------------------------
# BUILD PROTOTYPES
# -------------------------
prototypes = {}

for item in ["apple","banana","capsicum","cucumber","potato"]:

    fresh_list = []
    rotten_list = []

    for i, lbl in enumerate(labels):
        lbl_lower = lbl.lower()

        if item in lbl_lower:
            if "fresh" in lbl_lower:
                fresh_list.append(X_norm[i])
            else:
                rotten_list.append(X_norm[i])

    if len(fresh_list) > 0 and len(rotten_list) > 0:
        prototypes[item] = {
            "fresh": np.mean(fresh_list, axis=0),
            "rotten": np.mean(rotten_list, axis=0)
        }

print("Prototypes Ready")

# -------------------------
# IDENTIFICATION
# -------------------------
def optimize_and_identify(img):

    hand_feat = extract_handcrafted_features(img)
    deep_feat = extract_deep_features(img)

    fused = np.concatenate((hand_feat, deep_feat))

    fused_scaled = scaler.transform([fused])
    fused_norm = normalize(fused_scaled)[0]

    pred_class = svm_model.predict([fused_norm])[0]
    pred_label = le.inverse_transform([pred_class])[0].lower()

    if "apple" in pred_label: item = "Apple"
    elif "banana" in pred_label: item = "Banana"
    elif "capsicum" in pred_label: item = "Capsicum"
    elif "cucumber" in pred_label: item = "Cucumber"
    elif "potato" in pred_label: item = "Potato"
    else: item = "Unknown"

    return fused_norm, item

# -------------------------
# PROTOTYPE SCORING
# -------------------------
def compute_freshness_prototype(fused_norm, item):

    item_key = item.lower()

    if item_key not in prototypes:
        return 50, "Unknown"

    fresh_centroid = prototypes[item_key]["fresh"]
    rotten_centroid = prototypes[item_key]["rotten"]

    d_fresh = np.linalg.norm(fused_norm - fresh_centroid)
    d_rotten = np.linalg.norm(fused_norm - rotten_centroid)

    if (d_fresh + d_rotten) == 0:
        score = 50
    else:
        score = (d_rotten / (d_fresh + d_rotten)) * 100

    score = np.clip(score, 0, 100)

    if score < 25:
        category = "Fully Rotten"
    elif score < 50:
        category = "Rotten"
    elif score < 75:
        category = "Fresh"
    else:
        category = "Fully Fresh"

    return round(score,2), category

# -------------------------
# TEST
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\download.jpg"

img = cv2.imread(img_path)

if img is not None:

    fused_norm, item = optimize_and_identify(img)

    score, category = compute_freshness_prototype(fused_norm, item)

    print("\nItem Detected:", item)
    print("Freshness Score:", score,"%")
    print("Category:", category)

else:
    print("Image not found")

Loading dataset...
Dataset Loaded
Training Done
Prototypes Ready

Item Detected: Cucumber
Freshness Score: 52.63 %
Category: Fresh


In [2]:
# -------------------------
# TEST IMAGE
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\rotten-potatoes-cropped-against-white-260nw-2592664299.webp"

img = cv2.imread(img_path)

if img is not None:

    # Step 1: Extract features + identify item
    fused_norm, item = optimize_and_identify(img)

    # Step 2: Compute freshness score
    score, category = compute_freshness_prototype(fused_norm, item)

    # Step 3: Output
    print("\nItem Detected:", item)
    print("Freshness Score:", score, "%")
    print("Category:", category)

else:
    print("Image not found")


Item Detected: Potato
Freshness Score: 48.05 %
Category: Rotten


In [3]:
# -------------------------
# TEST IMAGE
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\Screen Shot 2018-06-08 at 4.59.36 PM.png"

img = cv2.imread(img_path)

if img is not None:

    # Step 1: Extract features + identify item
    fused_norm, item = optimize_and_identify(img)

    # Step 2: Compute freshness score
    score, category = compute_freshness_prototype(fused_norm, item)

    # Step 3: Output
    print("\nItem Detected:", item)
    print("Freshness Score:", score, "%")
    print("Category:", category)

else:
    print("Image not found")


Item Detected: Apple
Freshness Score: 52.51 %
Category: Fresh


In [4]:
# -------------------------
# TEST IMAGE
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\orange.jpg"

img = cv2.imread(img_path)

if img is not None:

    # Step 1: Extract features + identify item
    fused_norm, item = optimize_and_identify(img)

    # Step 2: Compute freshness score
    score, category = compute_freshness_prototype(fused_norm, item)

    # Step 3: Output
    print("\nItem Detected:", item)
    print("Freshness Score:", score, "%")
    print("Category:", category)

else:
    print("Image not found")


Item Detected: Apple
Freshness Score: 50.21 %
Category: Fresh


In [5]:
# -------------------------
# TEST IMAGE
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\potato.jpg"

img = cv2.imread(img_path)

if img is not None:

    # Step 1: Extract features + identify item
    fused_norm, item = optimize_and_identify(img)

    # Step 2: Compute freshness score
    score, category = compute_freshness_prototype(fused_norm, item)

    # Step 3: Output
    print("\nItem Detected:", item)
    print("Freshness Score:", score, "%")
    print("Category:", category)

else:
    print("Image not found")


Item Detected: Potato
Freshness Score: 53.32 %
Category: Fresh


In [6]:
# -------------------------
# TEST IMAGE
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\100-percent-fresh-natural-pure-organic-round-fresh-potato-for-cooking-use-857.jpg"

img = cv2.imread(img_path)

if img is not None:

    # Step 1: Extract features + identify item
    fused_norm, item = optimize_and_identify(img)

    # Step 2: Compute freshness score
    score, category = compute_freshness_prototype(fused_norm, item)

    # Step 3: Output
    print("\nItem Detected:", item)
    print("Freshness Score:", score, "%")
    print("Category:", category)

else:
    print("Image not found")


Item Detected: Potato
Freshness Score: 50.92 %
Category: Fresh


In [8]:
# -------------------------
# TEST IMAGE
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\images.jpg"

img = cv2.imread(img_path)

if img is not None:

    # Step 1: Extract features + identify item
    fused_norm, item = optimize_and_identify(img)

    # Step 2: Compute freshness score
    score, category = compute_freshness_prototype(fused_norm, item)

    # Step 3: Output
    print("\nItem Detected:", item)
    print("Freshness Score:", score, "%")
    print("Category:", category)

else:
    print("Image not found")


Item Detected: Potato
Freshness Score: 47.89 %
Category: Rotten


In [9]:
# -------------------------
# TEST IMAGE
# -------------------------
img_path = r"C:\Users\KIIT0001\Desktop\test_images\istockphoto-1475266930-612x612.jpg"

img = cv2.imread(img_path)

if img is not None:

    # Step 1: Extract features + identify item
    fused_norm, item = optimize_and_identify(img)

    # Step 2: Compute freshness score
    score, category = compute_freshness_prototype(fused_norm, item)

    # Step 3: Output
    print("\nItem Detected:", item)
    print("Freshness Score:", score, "%")
    print("Category:", category)

else:
    print("Image not found")


Item Detected: Cucumber
Freshness Score: 46.64 %
Category: Rotten
